In [1]:
# -*- coding: utf-8 -*-
import arcpy
import os
import shutil
from openpyxl import load_workbook
import time

# --- 1. 用户配置区域 ---
# 请在此处修改你的路径
SOURCE_GDB_FOLDER = r"D:\Pro Projects\CAD处理\CAD\1111-三年成果库\2024年"

# --- 配置结束 ---

def delete_features_by_prefix(gdb_path, field_name, prefix_to_delete):
    """
    在GDB的所有要素类中，删除指定字段中以特定前缀开头的记录。

    参数:
    - gdb_path: GDB的完整路径。
    - field_name: 用于筛选的字段名 (例如 "FEATID")。
    - prefix_to_delete: 要删除的记录的字段值前缀。
    """
    print(f"  - 开始删除 GDB '{os.path.basename(gdb_path)}' 中 '{field_name}' 字段以 '{prefix_to_delete}' 开头的记录...")
    try:
        # 获取 GDB 中的所有要素类路径（包括根目录下的和要素数据集内部的）
        fc_paths = []
        old_workspace = arcpy.env.workspace
        try:
            arcpy.env.workspace = gdb_path
            # 1. 获取根目录下的要素类
            for fc in arcpy.ListFeatureClasses() or []:
                fc_paths.append(os.path.join(gdb_path, fc))
            # 2. 获取要素数据集中的要素类
            for dataset in arcpy.ListDatasets(feature_type='feature') or []:
                dataset_path = os.path.join(gdb_path, dataset)
                arcpy.env.workspace = dataset_path
                for fc in arcpy.ListFeatureClasses() or []:
                    fc_paths.append(os.path.join(dataset_path, fc))
        finally:
            arcpy.env.workspace = old_workspace

        for fc_path in fc_paths:
            fc_name = os.path.basename(fc_path)
            
            # 检查所需字段是否存在于要素类中（优先使用指定字段，如 FEATID，若不存在则尝试使用 XZQDM 作为备用字段）
            field_names = [f.name for f in arcpy.ListFields(fc_path)]
            actual_field = None
            if field_name in field_names:
                actual_field = field_name
            elif "XZQDM" in field_names:
                actual_field = "XZQDM"
                
            if not actual_field:
                # print(f"    - 警告: 在要素类 '{fc_name}' 中未找到字段 '{field_name}' 或 'XZQDM'，跳过。")
                continue

            # 使用 where_clause 高效筛选，避免遍历所有记录
            # LIKE 'prefix%' 语法用于查找以特定字符串开头的值
            where_clause = f"{arcpy.AddFieldDelimiters(fc_path, actual_field)} LIKE '{prefix_to_delete}%'"
            
            # 统计将要删除的要素数量
            count_before = int(arcpy.management.GetCount(fc_path)[0])
            features_to_delete = int(arcpy.management.GetCount(arcpy.management.MakeFeatureLayer(fc_path, "temp_layer", where_clause))[0])

            if features_to_delete > 0:
                print(f"    - 在 '{fc_name}' 中找到 {features_to_delete} 条记录待删除。")
                with arcpy.da.UpdateCursor(fc_path, [actual_field], where_clause) as cursor:
                    for row in cursor:
                        cursor.deleteRow()
                count_after = int(arcpy.management.GetCount(fc_path)[0])
                print(f"      成功删除。要素数从 {count_before} 变为 {count_after}。")
            # else:
            #     print(f"    - 在 '{fc_name}' 中未找到需要删除的记录。")

    except Exception as e:
        print(f"    - 删除记录时发生错误: {e}")
        arcpy.AddError(arcpy.GetMessages(2))


def preprocess_specific_gdbs(source_folder):
    """
    【新增功能】
    在主流程开始前，对特定的GDB进行复制、重命名和数据清理。
    """
    print("\n--- 开始执行特定的GDB预处理任务 ---")
    
    # 定义要处理的GDB和相关操作
    gdb_operations = {
        "510117郫都区.gdb": {
            "copy_to": "510120高新西区.gdb",
            "delete_prefix_in_original": "510120",
            "delete_prefix_in_copy": "510117"
        },
        "510185简阳市.gdb": {
            "copy_to": "510186东部新区.gdb",
            "delete_prefix_in_original": "510186",
            "delete_prefix_in_copy": "510185"
        }
    }

    for original_gdb_name, actions in gdb_operations.items():
        original_gdb_path = os.path.join(source_folder, original_gdb_name)
        new_gdb_path = os.path.join(source_folder, actions["copy_to"])

        if not arcpy.Exists(original_gdb_path):
            print(f"警告: 源GDB '{original_gdb_name}' 不存在，跳过相关处理。")
            continue

        # 1. 复制GDB
        print(f"正在复制 '{original_gdb_name}' 到 '{actions['copy_to']}'...")
        try:
            # 如果目标已存在，先删除，确保是全新复制
            if arcpy.Exists(new_gdb_path):
                print(f"  目标 '{actions['copy_to']}' 已存在，正在删除...")
                # 使用 shutil.rmtree 是因为 arcpy.Delete_management 有时会因锁问题失败
                shutil.rmtree(new_gdb_path)
            arcpy.Copy_management(original_gdb_path, new_gdb_path)
            print("  复制完成。")
        except Exception as e:
            print(f"  复制GDB时发生错误: {e}")
            arcpy.AddError(arcpy.GetMessages(2))
            continue # 如果复制失败，则跳过后续删除操作

        # 2. 删除原始GDB中的特定记录
        delete_features_by_prefix(original_gdb_path, "FEATID", actions["delete_prefix_in_original"])
        
        # 3. 删除新GDB中的特定记录
        delete_features_by_prefix(new_gdb_path, "FEATID", actions["delete_prefix_in_copy"])

    print("--- GDB预处理任务完成 ---\n")
def main():
    """
    主执行函数，协调整个工作流程。
    """
    start_time = time.time()
    
    # 0. 【新增步骤】执行特定的GDB预处理（复制、重命名、删除要素）
    preprocess_specific_gdbs(SOURCE_GDB_FOLDER)


    end_time = time.time()
    print(f"\n--- 所有任务处理完毕，总耗时: {end_time - start_time:.2f} 秒 ---")

if __name__ == '__main__':
    main()



--- 开始执行特定的GDB预处理任务 ---
正在复制 '510117郫都区.gdb' 到 '510120高新西区.gdb'...
  复制完成。
  - 开始删除 GDB '510117郫都区.gdb' 中 'FEATID' 字段以 '510120' 开头的记录...
    - 在 'CQJCFWA' 中找到 1 条记录待删除。
      成功删除。要素数从 35 变为 34。
    - 在 'DLTBBHP' 中找到 10 条记录待删除。
      成功删除。要素数从 27 变为 17。
    - 在 'DLTBBHA' 中找到 15 条记录待删除。
      成功删除。要素数从 115 变为 100。
    - 在 'WYCGP' 中找到 3 条记录待删除。
      成功删除。要素数从 12 变为 9。
    - 在 'XXP' 中找到 1 条记录待删除。
      成功删除。要素数从 83 变为 82。
    - 在 'XXA' 中找到 48 条记录待删除。
      成功删除。要素数从 235 变为 187。
    - 在 'YLJGP' 中找到 1 条记录待删除。
      成功删除。要素数从 15 变为 14。
    - 在 'YLJGA' 中找到 6 条记录待删除。
      成功删除。要素数从 30 变为 24。
    - 在 'FLJGA' 中找到 1 条记录待删除。
      成功删除。要素数从 3 变为 2。
    - 在 'FLJGP' 中找到 2 条记录待删除。
      成功删除。要素数从 26 变为 24。
    - 在 'WHHDP' 中找到 1 条记录待删除。
      成功删除。要素数从 62 变为 61。
    - 在 'WHHDA' 中找到 1 条记录待删除。
      成功删除。要素数从 4 变为 3。
    - 在 'GYSSP' 中找到 14 条记录待删除。
      成功删除。要素数从 151 变为 137。
    - 在 'GYSSA' 中找到 3 条记录待删除。
      成功删除。要素数从 15 变为 12。
    - 在 'GYYLDA' 中找到 57 条记录待删除。
      成功删除。要素数从 196 变为 139。
    - 在 'JT